# Proyecto final – Ingeniería Civil
**Integrantes:** Andrés Rincón, Santiago González y Jhon Varela  
**Docente:** Sergio Luis Lubo Argumedo

Notebook reproducible de preparación, análisis y validación.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT/'datos').exists(): ROOT = ROOT.parent
df = pd.read_csv(ROOT/'datos'/'dataset_original.csv')
df.shape, df.head()

In [ ]:
df.info()
df.isna().sum().sort_values(ascending=False)

## Limpieza y trazabilidad
Se convierten fechas, se crean indicadores de faltantes y se imputan los campos numéricos faltantes con la mediana. Los valores originales se conservan en `dataset_original.csv`.

In [ ]:
for c in ["fecha_inicio_proyecto","fecha_corte","fecha_fin_planeada"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")
missing_cols=["costo_real_periodo_cop","avance_real_pct_acum","consumo_combustible_l","emisiones_co2_kg","material_reciclado_pct"]
for c in missing_cols:
    df[c+"_faltante"] = df[c].isna().astype(int)
    df[c] = df[c].fillna(df[c].median())
df.duplicated("registro_id").sum()

## Indicadores derivados

In [ ]:
df["desviacion_costo_cop"]=df.costo_real_periodo_cop-df.costo_planeado_periodo_cop
df["sobrecosto_pct"]=df.desviacion_costo_cop/df.costo_planeado_periodo_cop*100
df["desviacion_avance_pp"]=df.avance_real_pct_acum-df.avance_planeado_pct_acum
df["desviacion_maquinaria_h"]=df.horas_maquinaria_reales-df.horas_maquinaria_planeadas
df["combustible_por_hora"]=df.consumo_combustible_l/df.horas_maquinaria_reales.replace(0,np.nan)
df["emisiones_por_hora"]=df.emisiones_co2_kg/df.horas_maquinaria_reales.replace(0,np.nan)
df["mes_corte"]=df.fecha_corte.dt.to_period("M").astype(str)
df[["desviacion_costo_cop","sobrecosto_pct","desviacion_avance_pp"]].describe()

## Análisis por tipo de obra y relación con precipitación

In [ ]:
resumen_tipo=df.groupby("tipo_obra").agg(registros=("registro_id","count"),sobrecosto_promedio_pct=("sobrecosto_pct","mean"),desviacion_avance_pp=("desviacion_avance_pp","mean")).sort_values("sobrecosto_promedio_pct",ascending=False)
display(resumen_tipo)
print("Correlación precipitación / desviación avance:",df[["precipitacion_mm","desviacion_avance_pp"]].corr().iloc[0,1])

## Validación y ajuste
El cálculo se realiza sobre los registros originales con costo planeado y real no nulos, sin imputación, para no alterar la métrica.

In [ ]:
raw=pd.read_csv(ROOT/"datos"/"dataset_original.csv")
v=raw[["costo_planeado_periodo_cop","costo_real_periodo_cop"]].dropna()
p=v.costo_planeado_periodo_cop.to_numpy(); r=v.costo_real_periodo_cop.to_numpy()
for f in [0.95,1.00,1.05,1.10]:
    print(f, np.mean(np.abs(r-p*f)))

In [ ]:
# Atípicos IQR: marcar, no borrar automáticamente
for c in ["costo_real_periodo_cop","horas_maquinaria_reales","precipitacion_mm","incidentes_seguridad"]:
    q1,q3=df[c].quantile([.25,.75]); iqr=q3-q1
    df[c+"_atipico_iqr"]=((df[c]<q1-1.5*iqr)|(df[c]>q3+1.5*iqr)).astype(int)
df.to_csv(ROOT/"datos"/"dataset_procesado.csv",index=False)
print("Dataset procesado exportado.")